# Hybrid Solar Irradiance Forecasting
Scalable notebook structure implementing:
1. SARIMAX with hour-of-day exogenous features.
2. Hybrid SARIMAX + LSTM residual model.
Designed so components can later be replaced with TCN, N-BEATS, Autoformer, etc.

## Project Structure

```
forecasting/
├── data/
├── preprocessing.py
├── features.py
├── models/
│   ├── sarimax_model.py
│   ├── residual_lstm.py
│   └── hybrid.py
├── training.py
├── evaluation.py
└── notebook.ipynb
```


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from statsmodels.tsa.statespace.sarimax import SARIMAX


## Data preparation

In [ ]:
# Expected dataframe:
# timestamp | irradiance

df = df.copy()
df = df.sort_values("timestamp")
df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour

encoder = OneHotEncoder(sparse_output=False)
hour = encoder.fit_transform(df[["hour"]])

hour_cols=[f"hour_{i}" for i in range(hour.shape[1])]
hour_df=pd.DataFrame(hour,columns=hour_cols,index=df.index)

df=pd.concat([df,hour_df],axis=1)
df=df.set_index("timestamp")


## SARIMAX model

In [ ]:
class SarimaxForecaster:

    def __init__(self,
                 order=(2,0,2),
                 seasonal_order=(2,0,2,24)):
        self.order=order
        self.seasonal_order=seasonal_order
        self.model=None

    def fit(self,y,exog):
        self.model=SARIMAX(
            y,
            exog=exog,
            order=self.order,
            seasonal_order=self.seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
        return self

    def forecast(self,steps,exog_future):
        return self.model.get_forecast(
            steps=steps,
            exog=exog_future
        ).predicted_mean

    @property
    def fitted(self):
        return self.model.fittedvalues


## Residual Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class ResidualDataset(Dataset):

    def __init__(self,residuals,exog,window):

        self.X=[]
        self.y=[]

        for i in range(len(residuals)-window):
            x=np.hstack([
                residuals[i:i+window,None],
                exog[i:i+window]
            ])
            self.X.append(x.astype(np.float32))
            self.y.append(np.float32(residuals[i+window]))

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]


## Residual LSTM

In [ ]:
import torch.nn as nn

class ResidualLSTM(nn.Module):

    def __init__(self,input_size,hidden=64):
        super().__init__()
        self.lstm=nn.LSTM(
            input_size,
            hidden,
            batch_first=True
        )
        self.head=nn.Linear(hidden,1)

    def forward(self,x):
        _,(h,_) = self.lstm(x)
        return self.head(h[-1])


## Training

In [ ]:
# 1. Fit SARIMAX
# sarimax.fit(train_y,train_exog)

# 2. residual = train_y - sarimax.fitted

# 3. Train ResidualLSTM on residual dataset

# 4. Forecast:
# final = sarimax_forecast + lstm_residual_forecast


## Future Extensions

- Replace one-hot hour with sin/cos encoding.
- Add weather forecasts.
- Replace SARIMAX by Prophet/N-BEATS/Autoformer.
- Multi-horizon direct forecasting.
- Hyperparameter optimization with Optuna.
- MLflow experiment tracking.
- ONNX export.
